# Module 4 | Class 4 — SVM vs KNN Showdown

**Objective:** Train SVM and KNN classifiers on the Telco Churn dataset, experiment with hyperparameters, and understand when to use each algorithm.

**Dataset:** Telco Customer Churn (Kaggle) — preprocessed the same way as Module 3.

In [1]:
!pip install kagglehub -q

## Task 1: Prepare and Scale the Data

In [2]:
import pandas as pd
import numpy as np
import time
import kagglehub
import os

path = kagglehub.dataset_download("blastchar/telco-customer-churn")
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))

# Same preprocessing as Module 3 / Module 4
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

cat_cols = df.select_dtypes(include='object').columns.drop('customerID')
df_encoded = pd.get_dummies(df.drop('customerID', axis=1), columns=cat_cols, drop_first=True)

X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

print("X shape:", X.shape)

Using Colab cache for faster access to the 'telco-customer-churn' dataset.
X shape: (7043, 30)


/tmp/ipykernel_928/1647670313.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (5634, 30), Test: (1409, 30)


In [4]:
from sklearn.preprocessing import StandardScaler

# Both SVM and KNN are distance-based, so scaling is essential
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

**Note on dataset size:** the Telco dataset (~7,000 rows) is small enough that SVM with an RBF kernel should train in a reasonable time. If it takes too long in your environment, reduce the training set to 5,000 rows, e.g. `X_train_s, y_train = X_train_s[:5000], y_train[:5000]`, and note this in your submission.

## Task 2: Train an SVM Classifier

In [5]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report

start = time.time()
svm = SVC(kernel='rbf', random_state=42)
svm.fit(X_train_s, y_train)
svm_time = time.time() - start

y_pred_svm = svm.predict(X_test_s)
print(f"SVM Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}")
print(f"SVM F1: {f1_score(y_test, y_pred_svm):.4f}")
print(f"SVM Training Time: {svm_time:.2f}s")

SVM Accuracy: 0.7928
SVM F1: 0.5562
SVM Training Time: 3.66s


## Task 3: Train KNN with K=5

In [6]:
from sklearn.neighbors import KNeighborsClassifier

start = time.time()
knn5 = KNeighborsClassifier(n_neighbors=5)
knn5.fit(X_train_s, y_train)
knn5_time = time.time() - start

y_pred_knn5 = knn5.predict(X_test_s)
print(f"KNN (K=5) Accuracy: {accuracy_score(y_test, y_pred_knn5):.4f}")
print(f"KNN (K=5) F1: {f1_score(y_test, y_pred_knn5):.4f}")
print(f"KNN (K=5) Training Time: {knn5_time:.2f}s")

KNN (K=5) Accuracy: 0.7473
KNN (K=5) F1: 0.5123
KNN (K=5) Training Time: 0.00s


## Task 4: Experiment with Different K Values

In [7]:
k_results = []

for k in [3, 5, 10]:
    start = time.time()
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_s, y_train)
    k_time = time.time() - start

    y_pred_k = knn.predict(X_test_s)
    acc = accuracy_score(y_test, y_pred_k)
    f1 = f1_score(y_test, y_pred_k)

    print(f"K={k} -> Accuracy: {acc:.4f}, F1: {f1:.4f}, Time: {k_time:.2f}s")
    k_results.append({'K': k, 'Accuracy': acc, 'F1': f1, 'Training Time (s)': k_time})

K=3 -> Accuracy: 0.7438, F1: 0.5128, Time: 0.01s
K=5 -> Accuracy: 0.7473, F1: 0.5123, Time: 0.00s
K=10 -> Accuracy: 0.7729, F1: 0.5266, Time: 0.01s


In [8]:
k_table = pd.DataFrame(k_results)
k_table

,K,Accuracy,F1,Training Time (s)
0,3,0.743790,0.512821,0.005093
1,5,0.747339,0.512329,0.003626
2,10,0.772889,0.526627,0.005711


**Which K performed best?** [Look at the table above and identify the K with the highest F1 (F1 is usually the better metric here since churn datasets are imbalanced). Lower K values (like K=3) make the model more sensitive to noise in individual nearby points — this can lead to overfitting, where the decision boundary follows small quirks in the training data. Higher K values (like K=10) average over more neighbors, producing smoother, more general decision boundaries — but pushed too high, this can underfit and blur real distinctions between classes. State which K in your table gave the best F1 and briefly explain whether that fits the overfitting/underfitting pattern.]

*(Replace the bracketed text with your own conclusion based on your actual K-value table.)*

## Task 5: SVM vs Best KNN — Full Comparison

In [9]:
# Fill in best_k with whichever K value had the highest F1 in Task 4
best_k = k_table.loc[k_table['F1'].idxmax(), 'K']
best_k_row = k_table.loc[k_table['F1'].idxmax()]

summary = pd.DataFrame({
    'Model': ['SVM (RBF)', f'KNN (K={int(best_k)})'],
    'Accuracy': [accuracy_score(y_test, y_pred_svm), best_k_row['Accuracy']],
    'F1': [f1_score(y_test, y_pred_svm), best_k_row['F1']],
    'Training Time (s)': [svm_time, best_k_row['Training Time (s)']]
})
summary

,Model,Accuracy,F1,Training Time (s)
0,SVM (RBF),0.792761,0.556231,3.660342
1,KNN (K=10),0.772889,0.526627,0.005711


**Note on "training time":** KNN's `.fit()` call is misleadingly fast — it isn't really learning anything, just storing the training data in memory. The real computational cost happens at *prediction* time, when it calculates the distance from each test point to every stored training point. SVM, by contrast, does real optimization work during `.fit()` (finding the maximum-margin hyperplane), which is why its training time is typically much higher — but its prediction step is fast, since it only needs to check a new point against the learned support vectors.

## Task 6: Discussion

**When would you choose KNN over SVM in a real project?**

[Write 3-5 sentences here based on your own results and the following considerations, then replace this bracketed placeholder:]

KNN is a reasonable choice when the dataset is small to medium-sized, the feature space has relatively low dimensionality, and interpretability matters — it's easy to explain a KNN prediction to a non-technical stakeholder as "this customer looks similar to these 5 other customers, most of whom churned." SVM tends to be preferred when the dataset has many features (SVM handles high-dimensional spaces well, while KNN suffers from the curse of dimensionality — in high dimensions, the notion of "nearest neighbor" becomes less meaningful because distances between points become more uniform). Training time is also a practical factor: SVM is slower to train but fast at prediction time, making it well-suited to situations where the model is trained once and then used repeatedly for many predictions. KNN is the opposite — near-instant to "train" (it just stores data) but potentially slow at prediction time on large datasets, since every prediction requires scanning the stored training set. For a production system serving many real-time predictions per second on a large customer base, SVM's prediction speed would likely be preferred; for quick prototyping or smaller datasets where explainability matters, KNN's simplicity is an advantage.

*(Adjust this paragraph to reflect your own actual timing and accuracy results from the tables above.)*